# Importation des modules

In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels

In [3]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


# Chargement des données
diviser les dépenses de santé par le PIB pour les avoir en %  
rajouter les données de dépenses de santé pour 1994 - 2004 en regardant sur les données de l'ocde  
rajouter la base mortalite  
faire le tri dans les pays de la bdd nb de médecins  


In [4]:
def load_data() :
    #PIB, pop_65, pop_tot, densite_medicale, out_of_pocket, mortalite :  #à remplacer par load_data() quand ce sera codé

    depenses_sante_vol = pd.read_excel(os.path.join("data", "Depenses_sante_en_volume.xlsx"))
    depenses_sante_PIB = pd.read_excel(os.path.join("data", "Depenses_Sante_PIB.xlsx"))
    pib = pd.read_excel(os.path.join("data", "PIB.xlsx"))
    pib_par_habitant = pd.read_excel(os.path.join("data", "PIB_par_habitant.xlsx"))
    pop_tot = pd.read_excel(os.path.join("data", "Pop_tot.xlsx"))
    pop65 = pd.read_excel(os.path.join("data", "Pop+65ans.xlsx"))
    part_pop65 = pd.read_excel(os.path.join("data", "part_pop_plus_65.xlsx"))
    pop_par_age = pd.read_excel(os.path.join("data", "Population_par_age_tranches5ans.xlsx"))
    
    return depenses_sante_vol, depenses_sante_PIB, pib, pib_par_habitant, pop_tot, pop65, part_pop65, pop_par_age

depenses_sante_vol, depenses_sante_PIB, pib, pib_par_habitant, pop_tot, pop65, part_pop65, pop_par_age = load_data()

In [5]:
pop_par_age

,Pays / Année,1994,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,1995,Unnamed: 9,...,Unnamed: 208,Unnamed: 209,Unnamed: 210,2024,Unnamed: 212,Unnamed: 213,Unnamed: 214,Unnamed: 215,Unnamed: 216,Unnamed: 217
0,Pays,65-69,70-74,75-79,80-84,85-89,90-94,95-99,65-69,70-74,...,85-89,90-94,95-99,65-69,70-74,75-79,80-84,85-89,90-94,95-99
1,Belgique,516500,437696,239853,219591,113613,37264,6584,515152,459282,...,213543,102182,24143,670050,567238,469003,300455,217679,104625,25056
2,Tchéquie,483375,398098,181648,184782,74179,18372,2255,477667,417611,...,133551,54609,9930,637575,612030,495745,287635,137279,55688,10459
3,Danemark,229316,209061,160384,116837,60090,20942,4056,224767,210865,...,89405,34976,9235,327014,295938,288628,179602,94445,35434,9215
4,Allemagne,3901183,3346660,1849688,1955113,968600,289132,0,3986724,3436438,...,1873598,655537,153543,5180675,4388965,3097954,3196790,2022499,647607,156910
5,Estonie,72541,47519,30244,25412,0,0,0,73213,50539,...,24990,10399,2203,81897,70199,49875,40095,26338,10480,2363
6,Irlande,127211,113768,81741,54389,22838,7196,0,126718,113455,...,56698,23559,5821,250313,213704,167279,105484,56866,25451,6850
7,Grèce,573390,366984,281633,209192,101287,30656,6581,597712,386340,...,255770,109089,24487,0,0,0,0,0,0,0
8,Espagne,1913565,1532901,1065040,755962,376714,120640,17929,1940045,1607984,...,985796,481419,121202,2729694,2276017,1972223,1353171,950358,501427,129567
9,France,2729115,2320871,1223222,1322641,723772,265067,50037,2739415,2484162,...,1338512,702118,199032,3933254,3695325,2967945,1834912,1363768,735411,213345


# Construction du Time-to-Death

In [5]:
def time_to_death(mortalite, pop) :
    # pour pop, à voir si on prend pop_tot, pop_65 ou une population plus âgée encore
    # ou faire une somme pondérée des mortalités pour les 65-70, 70-75, etc
    # fait par chatgpt, à revoir en fonction de la structure des données
    df = mortalite.merge(
        pop_tot,
        on=["country", "year", "age"],
        how="inner"
    )

    df["weighted_mortality"] = df["mortality_rate"] * df["population"]
    ttd = (
        df.groupby(["country", "year"])["weighted_mortality"]
          .sum()
          .reset_index(name="TTD")
    )
    return ttd

#pop = pop_65   # ou pop_80, ou pop_tot...
#ttd = time_to_death(mortalite, pop)

# Mise en forme du panel

In [13]:
def preparer_donnees(fichier, var) :   #var est un str indiquant le nom de la variable, comme "PIB" ou "depenses"
    df = fichier.copy()
    df = df.replace(':', np.nan)
    df = pd.melt(df, id_vars=['TIME'], var_name='Year', value_name=var)
    df = df.rename(columns={'TIME': 'Country'})
    df['Year'] = df['Year'].astype(int)  # S'assurer que l'année est un entier
    df = df.set_index(['Country', 'Year']).sort_index()
    return df

In [ ]:
panel_depenses_sante_PIB = preparer_donnees(depenses_sante_PIB, "Depenses de sante en % du PIB")
panel_PIB_par_habitant = preparer_donnees(pib_par_habitant, "PIB par habitant")
panel = pd.merge(panel_depenses_sante_PIB, panel_PIB_par_habitant, on=['Country', 'Year'], how='outer')
panel_pop65 = preparer_donnees(part_pop65, "Part des +65 ans")
panel = pd.merge(panel, panel_pop65, on=['Country', 'Year'], how='outer')


# 3. Structuration en Panel (MultiIndex)
df_panel = df_final.set_index(['Country', 'Year']).sort_index()

In [7]:
def panel() :
    #construction du panel contenant toutes les données dont on a besoin dans un seul panel
    #panel = (
    #    depenses_sante.merge(pop_65, on=["country", "year"])
    #       .merge(pib, on=["country", "year"])
    #       .merge(densite_medicale, on=["country", "year"])
    #       .merge(oop, on=["country", "year"])
    #       .merge(ttd, on=["country", "year"])
    #)

    #panel = panel.sort_values(["country", "year"])

    panel = depenses_sante_PIB.copy()
    
    # 2. Transformation du format Large (Wide) vers Long
    # On garde 'TIME' comme identifiant et on bascule toutes les années dans une seule colonne
    panel = pd.melt(panel, id_vars=['TIME'], var_name='Year', value_name='Health_Exp')
    
    # 3. Nettoyage et Renommage
    panel = panel.rename(columns={'TIME': 'Country'})
    panel['Year'] = panel['Year'].astype(int)  # S'assurer que l'année est un entier
    
    # 4. Définition du MultiIndex (Entité, Temps)
    # C'est cette étape qui crée la structure "Panel" officielle
    panel = panel.set_index(['Country', 'Year']).sort_index()
    
    return panel

#panel = panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd)

# Régression et GMM